# 04 - Schema 设计与数据操作（Schema & Data Manipulation）
> Harvard CS50’s Intro to Databases with SQL  |  课程时间：2:52:56 – 5:25:20

这章是数据库的「建筑课」—— 你不再只是查询别人建好的表，而是**从零设计、建表、插数据、改数据、删数据**。

## 学习路线

| # | 内容 |
|---|------|
| 4.1 | GROUP BY & HAVING — 分组聚合 |
| 4.2 | Schema 设计 — 从需求到表结构 |
| 4.3 | 范式化 — 1NF / 2NF / 3NF |
| 4.4 | 建表 & 数据类型 |
| 4.5 | 约束 — NOT NULL / UNIQUE / CHECK / DEFAULT |
| 4.6 | ALTER TABLE — 修改表结构 |
| 4.7 | 外键约束 — CASCADE / SET NULL / RESTRICT |
| 4.8–4.11 | CRUD 操作 — INSERT / DELETE / UPDATE |

> ⚠️ 先运行下面的 cell 创建练习环境！

In [ ]:
import sqlite3
import csv, io

def sql(query):
    statements = [s.strip() for s in query.strip().split(';') if s.strip()]
    if not statements:
        return
    result = None
    for stmt in statements:
        cursor = conn.execute(stmt)
        if cursor.description:
            result = cursor.fetchall()
            columns = [d[0] for d in cursor.description]
    conn.commit()
    if result is None:
        print('✅ Done')
        return
    if not result:
        print('(empty)')
        return
    col_widths = [len(c) for c in columns]
    for row in result:
        for i, val in enumerate(row):
            col_widths[i] = max(col_widths[i], len(str(val)))
    header = ' | '.join(c.ljust(col_widths[i]) for i, c in enumerate(columns))
    sep = '-+-'.join('-' * col_widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for row in result:
        print(' | '.join(str(v).ljust(col_widths[i]) for i, v in enumerate(row)))
    print(f'\n({len(result)} row{"s" if len(result) != 1 else ""})')

# 查看表结构
def describe(table_name):
    print(f'\n📋 {table_name} 表结构：')
    print(f'   {"列名":15s} {"类型":10s} {"约束"}')
    print(f'   {"-"*15} {"-"*10} {"-"*20}')
    for row in conn.execute(f"PRAGMA table_info('{table_name}')"):
        constraints = []
        if row[3]: constraints.append('NOT NULL')
        if row[5]: constraints.append('PRIMARY KEY')
        constr = ', '.join(constraints) if constraints else ''
        print(f'   {row[1]:15s} {row[2]:10s} {constr}')
    fks = conn.execute(f"PRAGMA foreign_key_list('{table_name}')").fetchall()
    for fk in fks:
        print(f'   → FK: {fk[3]} → {fk[2]}({fk[4]})  ON DELETE {fk[6]}')


conn = sqlite3.connect('books_ch4.db')
conn.execute("PRAGMA foreign_keys = ON")

# 创建练习用的初始数据
conn.executescript('''
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS authors;

CREATE TABLE authors (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birth_year INTEGER,
    country TEXT
);

CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    year INTEGER,
    pages INTEGER,
    rating REAL,
    genre TEXT,
    author_id INTEGER,
    FOREIGN KEY (author_id) REFERENCES authors(id)
);

INSERT INTO authors VALUES
    (1,'Stephen King',1947,'USA'),
    (2,'Andy Weir',1972,'USA'),
    (3,'Gillian Flynn',1971,'USA'),
    (4,'J.K. Rowling',1965,'UK'),
    (5,'George Orwell',1903,'UK'),
    (6,'Tara Westover',1986,'USA'),
    (7,'James Clear',1986,'USA'),
    (8,'Frank Herbert',1920,'USA');

INSERT INTO books VALUES
    (1,'The Shining',1977,447,4.2,'Horror',1),
    (2,'It',1986,1138,4.3,'Horror',1),
    (3,'The Stand',1978,823,4.1,'Horror',1),
    (4,'The Martian',2011,369,4.4,'Science Fiction',2),
    (5,'Project Hail Mary',2021,496,4.5,'Science Fiction',2),
    (6,'Gone Girl',2012,432,4.1,'Mystery',3),
    (7,'Sharp Objects',2006,254,4.0,'Mystery',3),
    (8,'Harry Potter 1',1997,223,4.4,'Fantasy',4),
    (9,'Harry Potter 2',1998,251,4.4,'Fantasy',4),
    (10,'1984',1949,328,4.5,'Fiction',5),
    (11,'Animal Farm',1945,112,4.2,'Fiction',5),
    (12,'Educated',2018,334,4.5,'Nonfiction',6),
    (13,'Atomic Habits',2018,320,4.4,'Nonfiction',7),
    (14,'Dune',1965,688,4.3,'Science Fiction',8);
''')

print('✅ 数据库就绪')
print(f'   authors: {conn.execute("SELECT COUNT(*) FROM authors").fetchone()[0]} 人')
print(f'   books:   {conn.execute("SELECT COUNT(*) FROM books").fetchone()[0]} 本')

---
## 4.1 GROUP BY & HAVING

`GROUP BY` 把数据按分类分别汇总。`HAVING` 过滤汇总后的结果。

In [ ]:
# 不分组 vs 分组
print('不分组——全部数据汇总成一行：')
sql('SELECT COUNT(*) AS total, ROUND(AVG(rating),2) AS avg_r FROM books;')

print('分组——每个 genre 各一行：')
sql('''SELECT genre, COUNT(*) AS cnt, ROUND(AVG(rating),2) AS avg_r
FROM books
GROUP BY genre
ORDER BY cnt DESC;''')

In [ ]:
# HAVING：过滤分组结果（只留下 ≥ 2 本的类型）
sql('''SELECT genre, COUNT(*) AS cnt, ROUND(AVG(rating),2) AS avg_r
FROM books
GROUP BY genre
HAVING COUNT(*) >= 2
ORDER BY avg_r DESC;''')

In [ ]:
# WHERE + GROUP BY + HAVING + ORDER BY 全家福
sql('''SELECT genre, COUNT(*) AS cnt, ROUND(AVG(rating),2) AS avg_r,
          ROUND(AVG(pages),0) AS avg_pages
FROM books
WHERE year >= 1990            -- ① 先过滤行：只要 1990 年后的
GROUP BY genre                -- ② 再分组
HAVING COUNT(*) >= 2          -- ③ 再过滤组：只要 ≥ 2 本的
ORDER BY avg_r DESC;          -- ④ 最后排序
''')

### 执行顺序回顾

```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
                  ↑                  ↑
             WHERE 在这里         HAVING 在这里
            （不能有聚合）        （可以有聚合）
```

In [ ]:
# 多列分组：按 genre + author_id 分组
sql('''SELECT genre, author_id, COUNT(*) AS cnt, ROUND(AVG(rating),2) AS avg_r
FROM books
GROUP BY genre, author_id
ORDER BY genre, cnt DESC;''')

### ✏️ 自己动手

按作者分组，统计每人写了几本书、平均评分，只看写了 ≥ 2 本的作者。

In [ ]:
sql("")

---
## 4.2 Schema 设计

假设我们要设计一个**在线书店**数据库。从需求到建表：

In [ ]:
# 设计一个新的数据库：bookstore
# 先销毁旧的，从头开始
conn.executescript('''
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS authors;
DROP TABLE IF EXISTS publishers;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS customers;
''')

print('旧表已清除，现在从零设计 ↓')

### 设计步骤

```
需求：在线书店
   - 有书（书名、价格、ISBN）
   - 书有作者（名字、国籍）
   - 书有出版社（名称、国家）
   - 有顾客（姓名、邮箱）
   - 顾客可以下单买书

识别实体 →
   authors    (id, name, country)
   books      (id, title, price, isbn, author_id, publisher_id)
   publishers (id, name, country)
   customers  (id, name, email)
   orders     (id, customer_id, order_date)       ← 一个订单多本书
   order_items(id, order_id, book_id, quantity)   ← N:M 联结表

关系：
   authors 1:N books
   publishers 1:N books
   customers 1:N orders
   orders N:M books (通过 order_items)
```

---
## 4.3 范式化 — 对比实验

先看一张'坏表'，再拆成规范化的表。

In [ ]:
# 非规范化表（违反 3NF）：author_name 和 author_country 依赖于 author_id
# 而不是直接依赖于 book_id
conn.execute('''CREATE TABLE bad_books (
    id INTEGER PRIMARY KEY,
    title TEXT,
    author_name TEXT,
    author_country TEXT
);''')

conn.executemany('INSERT INTO bad_books VALUES (?,?,?,?)', [
    (1, 'The Shining',   'Stephen King', 'USA'),
    (2, 'It',            'Stephen King', 'USA'),  -- 重复！
    (3, 'The Martian',   'Andy Weir',    'USA'),
    (4, 'Project Hail M.','Andy Weir',   'USA'),  -- 又重复！
])
conn.commit()

print('❌ 违反 3NF 的表：author_country 依赖于 author_name，而 author_name 不是主键')
sql('SELECT * FROM bad_books;')
print("\n如果改 Stephen King 的国籍... 要改两行，漏一行就不一致")

# 规范化：拆成两张表
conn.executescript('''
DROP TABLE IF EXISTS good_books;
DROP TABLE IF EXISTS good_authors;

CREATE TABLE good_authors (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    country TEXT
);

CREATE TABLE good_books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    author_id INTEGER,
    FOREIGN KEY (author_id) REFERENCES good_authors(id)
);

INSERT INTO good_authors VALUES (1, 'Stephen King', 'USA'), (2, 'Andy Weir', 'USA');
INSERT INTO good_books VALUES (1, 'The Shining', 1), (2, 'It', 1), (3, 'The Martian', 2), (4, 'Project Hail Mary', 2);
''')

print('\n✅ 规范化后：author 信息只存一处')
sql('''SELECT b.title, a.name, a.country
FROM good_books b JOIN good_authors a ON b.author_id = a.id;''')

---
## 4.4 建表 & 数据类型

从头构建在线书店的完整 Schema。

In [ ]:
# 清理 + 建完整 Schema
conn.executescript('''
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS authors;
DROP TABLE IF EXISTS publishers;
DROP TABLE IF EXISTS bad_books;
DROP TABLE IF EXISTS good_books;
DROP TABLE IF EXISTS good_authors;

-- 出版社
CREATE TABLE publishers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE,
    country TEXT,
    founded_year INTEGER
);

-- 作者
CREATE TABLE authors (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birth_year INTEGER,
    country TEXT
);

-- 书
CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    isbn TEXT UNIQUE,
    year INTEGER,
    pages INTEGER,
    price REAL,
    genre TEXT,
    author_id INTEGER NOT NULL,
    publisher_id INTEGER,
    FOREIGN KEY (author_id) REFERENCES authors(id),
    FOREIGN KEY (publisher_id) REFERENCES publishers(id)
);

-- 顾客
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    joined_date TEXT DEFAULT (date('now'))
);

-- 订单
CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL DEFAULT (date('now')),
    status TEXT DEFAULT 'pending',
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);

-- 订单项（N:M 联结表）
CREATE TABLE order_items (
    id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    book_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL DEFAULT 1,
    FOREIGN KEY (order_id) REFERENCES orders(id),
    FOREIGN KEY (book_id) REFERENCES books(id)
);
''')

print('✅ 在线书店 Schema 创建完成！')
for t in ['publishers','authors','books','customers','orders','order_items']:
    describe(t)

### 插入数据

In [ ]:
# 填充数据
sql('''INSERT INTO publishers (name, country, founded_year) VALUES
    ('Penguin', 'UK', 1935),
    ('HarperCollins', 'USA', 1989),
    ('Shueisha', 'Japan', 1925);''')

sql('''INSERT INTO authors (name, birth_year, country) VALUES
    ('Stephen King', 1947, 'USA'),
    ('J.K. Rowling', 1965, 'UK'),
    ('James Clear', 1986, 'USA'),
    ('Haruki Murakami', 1949, 'Japan');''')

sql('''INSERT INTO books (title, isbn, year, pages, price, genre, author_id, publisher_id) VALUES
    ('The Shining',      '978-0385121675', 1977, 447, 8.99,  'Horror',      1, 1),
    ('It',               '978-0450411434', 1986, 1138,12.99,  'Horror',      1, 2),
    ("Harry Potter 1",  '978-0747532743', 1997, 223,  9.99,  'Fantasy',     2, 1),
    ("Harry Potter 2",  '978-0747538486', 1998, 251,  9.99,  'Fantasy',     2, 1),
    ('Atomic Habits',   '978-0735211292', 2018, 320, 16.99,  'Nonfiction',  3, 2),
    ('Norwegian Wood',  '978-4062748698', 1987, 296, 12.99,  'Fiction',     4, 3);''')

sql('''INSERT INTO customers (name, email) VALUES
    ('Alice Smith', 'alice@email.com'),
    ('Bob Jones',   'bob@email.com'),
    ('Charlie Wu',  'charlie@email.com');''')

sql('''INSERT INTO orders (customer_id, order_date, status) VALUES
    (1, '2024-01-15', 'completed'),
    (1, '2024-03-20', 'shipped'),
    (2, '2024-02-10', 'completed'),
    (3, '2024-06-01', 'pending');''')

sql('''INSERT INTO order_items (order_id, book_id, quantity) VALUES
    (1, 1, 1), (1, 3, 2),
    (2, 5, 1),
    (3, 2, 1), (3, 4, 1),
    (4, 6, 1);''')

print('✅ 数据填充完成！看看各表的数据：')
print('\npublishers:');   sql('SELECT * FROM publishers;')
print('authors:');       sql('SELECT * FROM authors;')
print('books:');         sql('SELECT title, price, author_id, publisher_id FROM books;')
print('customers:');     sql('SELECT * FROM customers;')
print('orders:');        sql('SELECT * FROM orders;')
print('order_items:');   sql('SELECT * FROM order_items;')

### 验证设计：跨表查询

用前面学的 JOIN 来查「每个订单里有什么书」：

In [ ]:
sql('''SELECT o.id AS order_id, c.name AS customer, b.title,
          oi.quantity, o.status
FROM orders o
JOIN customers c ON o.customer_id = c.id
JOIN order_items oi ON o.id = oi.order_id
JOIN books b ON oi.book_id = b.id
ORDER BY o.id;''')

---
## 4.5 约束实战

约束是数据库帮你自动检查的规则。

In [ ]:
# UNIQUE：email 不能重复
try:
    sql("INSERT INTO customers (name, email) VALUES ('Fake', 'alice@email.com');")
except Exception as e:
    print(f'❌ UNIQUE 约束阻止了重复 email: {e}')

In [ ]:
# NOT NULL：不能为空
try:
    sql("INSERT INTO customers (name, email) VALUES (NULL, 'test@test.com');")
except Exception as e:
    print(f'❌ NOT NULL 约束阻止了空 name: {e}')

In [ ]:
# CHECK：创建一个有约束的表
conn.execute('''CREATE TABLE IF NOT EXISTS reviews (
    id INTEGER PRIMARY KEY,
    book_id INTEGER NOT NULL,
    score INTEGER NOT NULL CHECK (score >= 1 AND score <= 5),
    comment TEXT,
    FOREIGN KEY (book_id) REFERENCES books(id)
);''')

print('reviews 表已创建（score 必须在 1-5 之间）')

# 尝试插入超范围的评分
try:
    sql("INSERT INTO reviews (book_id, score) VALUES (1, 10);")
except Exception as e:
    print(f'❌ CHECK 约束阻止了 score=10: {e}')

In [ ]:
# DEFAULT：不指定值时自动填充
# orders 表的 order_date 有 DEFAULT date('now')
sql("INSERT INTO orders (customer_id, status) VALUES (3, 'pending');")
sql("SELECT id, customer_id, order_date, status FROM orders WHERE customer_id = 3;")

---
## 4.6 ALTER TABLE — 修改表结构

In [ ]:
# 查看当前 books 表结构
describe('books')

In [ ]:
# 添加列
sql("ALTER TABLE books ADD COLUMN language TEXT DEFAULT 'English';")
print('添加 language 列后：')
sql('SELECT id, title, language FROM books;')

In [ ]:
# 重命名列
sql("ALTER TABLE books RENAME COLUMN price TO price_usd;")
print('列名 price → price_usd：')
sql('SELECT id, title, price_usd FROM books;')

In [ ]:
# 删除列
sql("ALTER TABLE books DROP COLUMN language;")
describe('books')
print('language 列已删除')

---
## 4.7 外键约束深入

In [ ]:
# 测试 RESTRICT：有书的作者不能删
print('尝试删除 Stephen King（id=1，他还有 2 本书关联着）...')
try:
    sql("DELETE FROM authors WHERE id = 1;")
except Exception as e:
    print(f'❌ 外键约束阻止了删除: {e}')
    print('因为有 books 行引用了 author_id=1，默认 RESTRICT 行为')

In [ ]:
# 演示 CASCADE 的效果：重建 books 表，加上 ON DELETE CASCADE
conn.executescript('''
-- 先备份数据
CREATE TEMP TABLE books_backup AS SELECT * FROM books;

DROP TABLE books;

CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    isbn TEXT UNIQUE,
    year INTEGER,
    pages INTEGER,
    price_usd REAL,
    genre TEXT,
    author_id INTEGER NOT NULL,
    publisher_id INTEGER,
    FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE,
    FOREIGN KEY (publisher_id) REFERENCES publishers(id) ON DELETE SET NULL
);

INSERT INTO books SELECT * FROM books_backup;
DROP TABLE books_backup;
''')

print('现在 books 表的外键是 ON DELETE CASCADE')
print('\n删除前：')
sql("SELECT title FROM books WHERE author_id = 1;")

sql("DELETE FROM authors WHERE id = 1;")

print('\n删除 Stephen King 后 → 他的书自动消失：')
sql("SELECT COUNT(*) AS remaining FROM books WHERE author_id = 1;")

---
## 4.8–4.11 CRUD 完整操作

### INSERT

In [ ]:
# 插入单行
sql("INSERT INTO authors (name, birth_year, country) VALUES ('Chimamanda Adichie', 1977, 'Nigeria');")
sql("SELECT * FROM authors WHERE name = 'Chimamanda Adichie';")

In [ ]:
# 插入多行
sql('''INSERT INTO books (title, year, price_usd, genre, author_id, publisher_id) VALUES
    ('Half of a Yellow Sun', 2006, 14.99, 'Fiction',
        (SELECT id FROM authors WHERE name = 'Chimamanda Adichie'), 1),
    ('Americanah', 2013, 13.99, 'Fiction',
        (SELECT id FROM authors WHERE name = 'Chimamanda Adichie'), 1);''')
sql("SELECT title, year, price_usd FROM books WHERE price_usd > 13;")
print('\n💡 用子查询自动获取 author_id，不用手动查数字')

### 模拟 CSV 导入

In [ ]:
# 模拟从 CSV 文件导入数据
csv_data = """title,year,pages,price,genre,author_name
Kafka on the Shore,2002,467,13.49,Fiction,Haruki Murakami
1Q84,2009,928,16.99,Fiction,Haruki Murakami
Men Without Women,2014,228,11.99,Fiction,Haruki Murakami"""

reader = csv.DictReader(io.StringIO(csv_data))
inserted = 0
for row in reader:
    # 确保作者存在
    author = row['author_name']
    conn.execute("INSERT OR IGNORE INTO authors (name, country) VALUES (?, 'Japan');", (author,))
    author_id = conn.execute("SELECT id FROM authors WHERE name = ?;", (author,)).fetchone()[0]
    # 插入书
    conn.execute(
        "INSERT INTO books (title, year, pages, price_usd, genre, author_id) VALUES (?,?,?,?,?,?);",
        (row['title'], int(row['year']), int(row['pages']), float(row['price']), row['genre'], author_id)
    )
    inserted += 1

conn.commit()
print(f'✅ 从 CSV 导入了 {inserted} 本书')
sql('''SELECT b.title, a.name AS author, b.price_usd
FROM books b JOIN authors a ON b.author_id = a.id
WHERE a.name = 'Haruki Murakami';''')

### UPDATE

In [ ]:
# 修改单行
print('修改前：')
sql("SELECT title, price_usd FROM books WHERE title = 'Atomic Habits';")

sql("UPDATE books SET price_usd = 18.99 WHERE title = 'Atomic Habits';")

print('修改后：')
sql("SELECT title, price_usd FROM books WHERE title = 'Atomic Habits';")

In [ ]:
# 批量更新：所有 20 世纪之前的书价格减半
sql("UPDATE books SET price_usd = ROUND(price_usd * 0.5, 2) WHERE year < 2000;")
print('20 世纪之前的书降价后：')
sql('SELECT title, year, price_usd FROM books WHERE year < 2000 ORDER BY year;')

### DELETE

> ⚠️ 永远先用 SELECT 确认！

In [ ]:
# 安全删除流程：先 SELECT，确认无误再 DELETE
print('Step 1: 先看看哪些书会受影响——')
sql("SELECT id, title, price_usd FROM books WHERE price_usd < 5;")

print('\nStep 2: 确认后删除')
# 实际演示时跳过删，用注释表示
# sql("DELETE FROM books WHERE price_usd < 5;")
print('   DELETE FROM books WHERE price_usd < 5;  ← 确认后执行')

In [ ]:
# 实际删除一条
print('删除前订单 #4：')
sql('SELECT * FROM orders;')

sql("DELETE FROM order_items WHERE order_id = 4;")
sql("DELETE FROM orders WHERE id = 4;")

print('\n删除后：')
sql('SELECT * FROM orders;')

### ✏️ 自己动手

加一本你自己的书：书名叫你想叫的任何名字，价格 20.99，类型 Biography，作者就用 J.K. Rowling。

In [ ]:
sql("")

---
## 🎯 综合挑战

In [ ]:
# Q1：每个作者有多少本书？按数量降序排列
sql("")

In [ ]:
# Q2：找出有 ≥ 2 本书的类型，以及该类型的平均价格
sql("")

In [ ]:
# Q3：列出每个顾客下的订单数和总消费金额
#     提示：JOIN customers + orders + order_items + books
#           GROUP BY customer_id
sql("")

In [ ]:
# Q4：给 books 表加一个 CHECK 约束，保证 price 永远 > 0
#     提示：ALTER TABLE 加不了 CHECK... 只能重建表
#     所以这里写 CREATE TABLE 语句就好
sql("""
-- 在这里写 CREATE TABLE ... CHECK (price_usd > 0)
""")

In [ ]:
# Q5：日本的出版社（Shueisha）有哪些书？用 JOIN 查
sql("")

---
## ✅ 检查清单

- [ ] 能用 GROUP BY 分组聚合，配合多个聚合函数
- [ ] 理解 WHERE vs HAVING 的区别和执行时机
- [ ] 知道 FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY 执行顺序
- [ ] 能自己设计从需求到多表 Schema
- [ ] 理解 1NF / 2NF / 3NF 并识别反范式化问题
- [ ] 能用 CREATE TABLE 建表并选择正确的数据类型
- [ ] 会设约束：NOT NULL / UNIQUE / CHECK / DEFAULT / FOREIGN KEY
- [ ] 能用 ALTER TABLE 加减列、重命名
- [ ] 理解 ON DELETE CASCADE / SET NULL / RESTRICT
- [ ] 熟练掌握 INSERT / UPDATE / DELETE
- [ ] 知道安全删除流程：先 SELECT 确认，再 DELETE
- [ ] 能通过子查询自动解析外键 ID

---

> 📖 下一章：[05 - 高级特性](../05-advanced-features/) — Trigger、View、CTE、软删除、多对多